<a href="https://colab.research.google.com/github/2026SD4/26SD4_19_HATANAKA_MAO/blob/main/SD4AI%E6%BC%94%E7%BF%9202%E7%95%91%E4%B8%AD%E6%94%BF%E5%A4%AE_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U -q langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 16.6 MB/s eta 0:00:00


In [2]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
以下の料理のレシピを作成してください。
料理名:{dish}
""")

prompt_value = prompt.invoke({"dish":"チャーハン"})
print(prompt_value.to_string())



以下の料理のレシピを作成してください。
料理名:チャーハン



In [3]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system","ユーザーが入力した料理のレシピを考えてください。"),
    ("human","{dish}"),
])

prompt_value = prompt.invoke({"dish":"チャーハン"})
print(prompt_value.to_string())

System: ユーザーが入力した料理のレシピを考えてください。
Human: チャーハン


In [4]:
from typing import Optional
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 履歴用のリストに書くメッセージのクラスもimport
from langchain_core.messages import AIMessage, HumanMessage

# MessagesPlaceholderを使ってプロンプトを生成
prompt = ChatPromptTemplate.from_messages([
    ("system","あなたは親切で優秀なアシスタントです。"),
    # "chat_history"というキーワードのプレースホルダー
    MessagesPlaceholder("chat_history",Optional=True),
    ("human","{input}")
])

In [5]:
prompt_value = prompt.invoke({
    "chat_history":[
        HumanMessage(content="こんにちは！私はジョンです。映画俳優です。"),
        AIMessage(content="こんにちはジョンさん。はじめまして"),
    ],
    "input":"私の職業は何だったでしょうか。"
})

print(prompt_value)

messages=[SystemMessage(content='あなたは親切で優秀なアシスタントです。', additional_kwargs={}, response_metadata={}), HumanMessage(content='こんにちは！私はジョンです。映画俳優です。', additional_kwargs={}, response_metadata={}), AIMessage(content='こんにちはジョンさん。はじめまして', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='私の職業は何だったでしょうか。', additional_kwargs={}, response_metadata={})]


In [6]:
print(prompt_value.to_string())

System: あなたは親切で優秀なアシスタントです。
Human: こんにちは！私はジョンです。映画俳優です。
AI: こんにちはジョンさん。はじめまして
Human: 私の職業は何だったでしょうか。


In [7]:
!pip install -U -q langchainhub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.5 MB/s eta 0:00:00


In [9]:
import os
from google.colab import userdata
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")

In [12]:
from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

client = Client()
prompt = client.pull_prompt(
    "oshima/recipe",
    include_model = False,
    dangerously_pull_public_prompt = True
)

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(model="gemini-flash-latest",google_api_key=GOOGLE_API_KEY)
chain = prompt | model
response = chain.invoke({"dish":"チャーハン"})
print(response.content)

LangSmithError: Failed to GET /commits/oshima/recipe/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/oshima/recipe/latest', '{"error":"Forbidden"}\n')

### LangSmithなしでChatGoogleGenerativeAIを直接使用する

`LangSmith`を使用せず、`ChatGoogleGenerativeAI`を直接使用してプロンプトを作成し、応答を生成する方法を以下に示します。

まず、必要なライブラリをインポートし、`GOOGLE_API_KEY`を設定します。

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
from langchain_core.messages import AIMessage, HumanMessage

# Google API Keyを取得
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

次に、`ChatPromptTemplate`を定義し、`ChatGoogleGenerativeAI`モデルを初期化します。そして、それらを組み合わせてチェーンを作成します。

In [14]:
# ChatPromptTemplateの定義
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "あなたは親切で優秀なアシスタントです。料理のレシピ作成を支援します。"),
    ("human", "{dish}のレシピを作成してください。")
])

# ChatGoogleGenerativeAIモデルの初期化
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=GOOGLE_API_KEY)

# チェーンの作成
chain = prompt_template | model

最後に、チェーンを呼び出してレシピを生成します。

In [15]:
# チェーンの呼び出し
response = chain.invoke({"dish": "チャーハン"})
print(response.content)

GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.7-flash\nPlease retry in 32.08641154s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier',

In [16]:
from pydantic import BaseModel, Field
from typing import List

class Recipe(BaseModel):

  ingredients: List[str] = Field(description="料理の材料リスト")
  steps: List[str] = Field(description="調理手順のリスト")

In [17]:
from langchain_core.output_parsers import PydanticOutputParser
#Recipeクラス用のPydanticOutputParser
output_parser = PydanticOutputParser(pydantic_object=Recipe)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "料理の材料リスト", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "調理手順のリスト", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```


In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system","ユーザーが入力した料理のレシピを考えてください。 \n\n{format_instructions}"),
    ("human","{dish}")
])

prompt_with_parser = prompt.partial(format_instructions=format_instructions)

print( prompt_with_parser.to_json())

{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'], 'kwargs': {'input_variables': ['dish'], 'partial_variables': {'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"ingredients": {"description": "料理の材料リスト", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "調理手順のリスト", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}\n```'}, 'messages': [SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['f

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
model = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=GOOGLE_API_KEY, temperature=0.7)

structured_llm = model.with_structured_output( Recipe )
chain = prompt_with_parser | structured_llm

recipe_object = chain.invoke({"dish":"欧風カレー"})

print( type(recipe_object))
print( recipe_object )

GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.7-flash\nPlease retry in 40.211446871s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier'

In [ ]:
print("材料リスト:")
for ingredient in recipe_object.ingredients:
  print(ingredient)

print("\n調理手順:")
for steps in recipe_object.steps:
  print(steps)